In [2]:
import sys
print(sys.executable)

C:\Users\jugal\langchain-env\Scripts\python.exe


# Built-in tools

1) DuckDuckGoSearch:
2) ShellTool

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

In [8]:
search = DuckDuckGoSearchRun()
result = search.invoke("IPL news")

print(result)

C:\Users\jugal\langchain-env\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


The official IPL website with live scores, match updates, team rankings, all the latest news, and videos . Follow your favorite teams and players! Mar 28, 2011 · Stay updated with the latest from IPL 2025, including live scores, match schedules, results, team rosters, and in-depth news coverage. Get Live Cricket Scores, Scorecard, Schedules of International and Domestic cricket matches along with Latest News, Videos and ICC Cricket Rankings of Players on Cricbuzz. Explore the latest updates, news, and insights about IPL news. Stay informed with exclusive stories, match highlights, and detailed analysis related to IPL news. Follow IPL news on … Jan 17, 2025 · Get the latest news, match highlights, team updates, and in-depth analysis of the IPL season. Stay informed with exclusive content, player stats.


# Custom Tools:
There are many ways to create custom tools but as for now, we are learning the most simple way to create a tool. Later we will learn other ways as well to create complex tools

1) Using tool decorator
2) Using Structured tool (Pydantic for type validation)
3) Using BaseTool class

# 1. Using `tool` decorator

In [9]:
from langchain_core.tools import tool

In [30]:
# Step 1: Create a function

def multiply(a, b, c = 1):
    ''' Multiplies the given numbers'''
    return a*b*c

# Step 2: Add type hinting

def multiply(a:int, b:int, c:int = 1) ->int:
    ''' Multiplies the given numbers'''
    return a*b*c

# Step 3: Add tool decorator

@tool
def multiply(a:int, b:int, c:int = 1) ->int:
    ''' Multiplies the given numbers'''
    return a*b*c

In [31]:
# Since all the tools in langchain are Runnables, we can call them using invoke() method
multiply.invoke({'a': 5, 'b': 5})

25

In [37]:
@tool
def add(a:int, b:int, c:int=0) ->int:
    '''Adds the given numbers'''
    return a+b+c

In [38]:
add({'a': 2, 'b': 5, 'c':4})

11

In [41]:
print(add.name)
print(add.description)
print(add.args)

add
Adds the given numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}, 'c': {'default': 0, 'title': 'C', 'type': 'integer'}}


What LLM will see?
LLM will not see the tool, it will see the json schema given below

In [46]:
print(add.args_schema.model_json_schema())

{'description': 'Adds the given numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}, 'c': {'default': 0, 'title': 'C', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


# 2. Using StructuredTool

In [53]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Annotated, Optional

In [63]:
# Step 1: Create pydintic base class
class MultiplyInput(BaseModel):

    a: Annotated[int, Field(description = "First number to multiply")]
    b: Annotated[int, Field(description = "Second number to multiply")]
    c: Annotated[int, Field(description = "Default/ user_input third number to multiply")] = 1

# Create a function
def multiply_func(a:int, b:int, c= 1) ->int:
    return a*b*c

multiply_tool = StructuredTool.from_function(
    func = multiply_func,
    name = 'multiply',
    description = "Multiply the given numbers",
    args_schema = MultiplyInput
)

In [64]:
result = multiply_tool.invoke({'a': 5, 'b': 5})

In [65]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

25
multiply
Multiply the given numbers
{'a': {'description': 'First number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number to multiply', 'title': 'B', 'type': 'integer'}, 'c': {'default': 1, 'description': 'Default/ user_input third number to multoply', 'title': 'C', 'type': 'integer'}}


# 3) Using BaseTool

* `BaseTool` is the abstract base class for all the tools in Langchain.
* It defines the core structure and interface that any tool must follow.
* All the other tools like `@tool`, `StructuredTool` are built on the top of `BaseTool`.

In [5]:
from langchain_core.tools import BaseTool
from typing import Type, Annotated
from pydantic import BaseModel, Field


In [6]:
class MultiplySchema(BaseModel):
    a: Annotated[int, Field(description = "First integer input")]
    b: Annotated[int, Field(description = 'Second integer input')]
    c: Annotated[int, Field(description = 'Third integer inout', default = 1)]

In [8]:
class Multiply(BaseTool):
    name: str = "multiply"
    description: str = "Multiplies the given numbers"
    args_schema: Type[BaseModel] = MultiplySchema

    def _run(self, a:int, b:int, c:int = 1) ->int:
        return a*b*c
        

In [13]:
mult = Multiply()
result = mult.invoke({'a': 4, 'b': 5, 'c': 2})

print(result)
print()

print(mult.name)
print()

print(mult.args)
print()

print(mult.description)
print()

40

multiply

{'a': {'description': 'First integer input', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer input', 'title': 'B', 'type': 'integer'}, 'c': {'default': 1, 'description': 'Third integer inout', 'title': 'C', 'type': 'integer'}}

Multiplies the given numbers



# Note: In 90% cases, we use method 1 or method 2 to create tools in langchain.  

# Toolkit

# What is a toolkit?

While creating any agent we would create many tool. So langchain, for our convenience, provides the dunctionality of Toolkit.<br>
Toolkit is a collection of similar tools foe the user convenience.

For example: We have below tools <br>
1. GoogleDriveCreateFileTool
2. GoogleDriveSearchTool
3. GoogleDriveReadFileTool

In this case, we can create a toolkit named `GoogleDriveToolkit` and put all these tools in it.

In [15]:
from langchain_core.tools import tool

In [17]:
# Creating custom tools

@tool
def add(a:int, b:int, c:int) ->int:
    '''Add the given numbers'''
    
    return a+b+c

@tool
def multiply(a:int, b:int, c:int) ->int:
    '''Multiply the given numbers'''
    
    return a*b*c


# Creating a tool kit

class MathToolKit:
    def get_tools(self):
        return(add, multiply)

In [18]:
tk = MathToolKit()

tk.get_tools()

(StructuredTool(name='add', description='Add the given numbers', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001BE1090AD40>),
 StructuredTool(name='multiply', description='Multiply the given numbers', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001BE14549940>))

# Session Complete.

# In this session we learned to create tools. In next session we will learn how to integrate these tools with LLM.